# الدرس 04: تحديث فهرس المتجهات وإضافة مستندات جديدة
### Adding New Documents to an Existing Vector Store

في التطبيقات الواقعية، تُضاف مستندات وملفات جديدة باستمرار (تقارير، سياسات، أوراق بحثية جديدة).
إعادة بناء وتضمين قاعدة بيانات المتجهات من الصفر في كل مرة أمر مكلف مالياً وزمنياً وغير عملي.

في هذه المحاضرة (**المحاضرة 29**)، سنتعلم كيفية إضافة وتحديث مستندات جديدة مباشرة في فهرس `ChromaDB` القائم على القرص عبر دالة `add_documents`، والتحقق الفوري من قدرة الـ Retriever ومسار الـ RAG على الإجابة من المعرفة الجديدة.

### 1. استيراد المكتبات وتحديد المسار المركزي لقاعدة البيانات
**آلية العمل:**
نقوم بتحميل متغيرات البيئة واستيراد كائنات LangChain الأساسية:
- كائن `Document` لإنشاء أو تغليف المستندات مع بياناتها الوصفية (`Metadata`).
- `Chroma` و `HuggingFaceEmbeddings` لإدارة واستدعاء الفهرس المخزن على القرص.
- `RecursiveCharacterTextSplitter` لتقطيع المستندات الجديدة قبل الإضافة.
- `ChatGroq` و `LCEL` لتجربة الاستعلام التوليدي على المستند الجديد.

In [ ]:
# import standard libraries and set up paths
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized database directory
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# langchain imports
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All libraries imported successfully!")

### 2. الاتصال بفهرس المتجهات الموجود مسبقاً وفحص عدد السجلات
**آلية العمل:**
نقوم بفتح قاعدة بيانات `Chroma` الموجودة على القرص دون مسحها، ونستخدم ميثود `_collection.count()` لمعرفة عدد المتجهات الحالية قبل إضافة أي بيانات جديدة.

In [ ]:
# initialize embedding function matching existing collection
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect to the existing persistent vector store
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

initial_count = vector_store._collection.count()
print(f"Connected to existing vector store at: {db_dir}")
print(f"Current document count BEFORE update: {initial_count}")

### 3. إنشاء المستند الجديد وتزويده ببيانات وصفية (Metadata)
**آلية العمل:**
في المشاريع الحقيقية قد تقرأ ملف PDF أو Word جديداً. هنا نقوم بإنشاء كائن `Document` يحوي معرفة جديدة عن **التعلم المعزز (Reinforcement Learning)** لم تكن موجودة في الفهرس من قبل.
نضيف بيانات وصفية مثل `source` و `topic`، حيث تساعد الـ Metadata لاحقاً في التصفية (Filtering) وتتبع مصدر المعلومة بدقة.

In [ ]:
# define new document content about Reinforcement Learning
rl_content = (
    "Reinforcement learning (RL) is a subfield of machine learning where an autonomous agent learns to make optimal decisions "
    "by taking actions within an interactive environment to maximize cumulative reward. Key foundational concepts in RL include: "
    "1. Agent: The decision-maker and learner. "
    "2. Environment: Everything external that the agent interacts with. "
    "3. State (S): Current representation of the environment. "
    "4. Action (A): All possible decisions the agent can execute. "
    "5. Reward (R): Immediate numerical feedback signal returned by the environment. "
    "6. Policy (pi): The agent's strategy mapping states to actions. "
    "7. Value Function (V): The expected long-term future return from a given state."
)

# create Document instance with rich metadata
new_doc = Document(
    page_content=rl_content,
    metadata={
        "source": "manual_addition",
        "topic": "reinforcement_learning",
        "author": "AI Team",
        "version": 1.0
    }
)

print("New document created successfully!")
print(f"Metadata: {new_doc.metadata}")

### 4. تقطيع المستند الجديد إلى Chunks
**آلية العمل:**
قبل إدخال أي مستند لقاعدة المتجهات، يجب تمريره على الـ Text Splitter لتقسيمه إلى أجزاء بحجم مناسب للبحث والاسترجاع، مع توارث نفس الـ Metadata لكل قطعة تلقائياً.

In [ ]:
# split the new document into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
new_chunks = text_splitter.split_documents([new_doc])

print(f"New document was split into {len(new_chunks)} chunks.")
for idx, chunk in enumerate(new_chunks, start=1):
    print(f"  Chunk [{idx}] ({len(chunk.page_content)} chars): {chunk.page_content[:80]}...")

### 5. إضافة القطع الجديدة للفهرس القائم (`add_documents`)
**آلية العمل:**
نستخدم دالة `vector_store.add_documents(new_chunks)`.
تقوم الدالة داخلياً بما يلي:
1. توليد التضمينات (Embeddings) للقطع الجديدة فقط عبر النموذج.
2. إلحاق المتجهات الجديدة إلى جدول `SQLite` وفهرس `HNSW` على القرص الصلب مباشرة دون المساس بالمتجهات القديمة.
3. ترجع قائمة بالـ IDs الجديدة التي تم حفظها.

In [ ]:
# add new chunks directly into the existing vector store
added_ids = vector_store.add_documents(new_chunks)

updated_count = vector_store._collection.count()
print(f"Successfully added {len(added_ids)} new vectors!")
print(f"Document count AFTER update: {updated_count} (Increased by +{updated_count - initial_count})")

### 6. التحقق والاستعلام عبر مسار LCEL RAG الكامل
**آلية العمل:**
نربط المسترجع (`Retriever`) بسلسلة LCEL ونسأل سؤالاً عن **الموضوع الجديد المضاف (التعلم المعزز)** للتأكد من:
1. أن الفهرس المحدث استرجع قطع الـ Reinforcement Learning فورياً.
2. أن الـ LLM صاغ إجابة دقيقة ومعتمدة على السياق الجديد.

In [ ]:
# helper function to format context chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# create retriever with k=3
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# define prompt and llm
template = """You are a helpful assistant. Use ONLY the provided context to answer the question.
If the answer cannot be found in the context, say "I don't know."
Keep the answer concise and well-structured.

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# construct lcel rag pipeline
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# query the pipeline about newly added knowledge
query = "What are the key concepts in reinforcement learning?"
answer = rag_chain.invoke(query)

print(f"❓ Question: {query}\n")
print(f"💬 Answer from Updated Store:\n{answer}\n")

# inspect retrieved chunks to verify metadata
retrieved_chunks = retriever.invoke(query)
print("📚 Retrieved Chunks Verification:")
for i, doc in enumerate(retrieved_chunks, start=1):
    topic = doc.metadata.get("topic", "none")
    src = doc.metadata.get("source", "none")
    print(f"  [{i}] (Topic: {topic} | Source: {src}): {doc.page_content[:75]}...")

## 🎯 ملخص الدرس 04 (المحاضرة 29):
1. **`vector_store.add_documents(chunks)`**: هي الطريقة القياسية لتحديث فهارس المتجهات بإضافة مستندات جديدة فوراً دون إعادة بناء الفهرس.
2. **البيانات الوصفية (`Metadata`)**: تعطي كل قطعة بطاقة تعريفية (`source`, `topic`, `author`) تمكننا لاحقاً من إجراء عمليات تصفية دقيقة (Metadata Filtering).
3. **التكامل الفوري**: بمجرد استدعاء `add_documents`، يصبح الـ `Retriever` ومسار الـ RAG قادراً فوراً على استرجاع المعرفة المضافة والإجابة منها.

**المحاضرة التالية (30):** إضافة **ذاكرة المحادثة (`Conversational Memory`)** لمسار الـ RAG ليصبح شات بوت حقيقياً يتذكر سياق الأسئلة السابقة!